# Pantanal BirdCLEF 2026 — Data Processing & Two-Phase Training

**Phase 1**: Pre-train on short bird clips (XC / iNat) — head only, backbone frozen  
**Phase 2**: Fine-tune on labeled soundscape segments — full model, lower LR

In [2]:
import pandas as pd
import os
from sklearn.model_selection import train_test_split

SEED = 42

bird_audio_dir = "/home/users/ss1482/sangcs372final/Finalproject/birdtrain_wav"
soundscape_dir = "/home/users/ss1482/sangcs372final/Finalproject/soundtrain"

# =========================
# 1. LOAD TRAIN AUDIO METADATA
# =========================
df = pd.read_csv("train.csv")

# Strip the subdirectory prefix, then convert extension
df["filename"] = df["filename"].apply(lambda x: os.path.basename(x))  # '1161364/iNat1216197.ogg' → 'iNat1216197.ogg'
df["filename"] = df["filename"].str.replace(".ogg", ".wav", regex=False)

df["filepath"] = df["filename"].apply(lambda x: os.path.join(bird_audio_dir, x))
df = df[df["filepath"].apply(os.path.exists)].reset_index(drop=True)
print("Files found:", len(df))
# =========================
# 2. LOAD SOUNDSCAPE LABELS
# =========================
ss_df = pd.read_csv("train_soundscapes_labels.csv")

# Convert filenames
ss_df["filename"] = ss_df["filename"].str.replace(".ogg", ".wav", regex=False)

# Path to soundscape wavs
soundscape_dir = "/home/users/ss1482/sangcs372final/Finalproject/soundtrain"

# Build filepaths
ss_df["filepath"] = ss_df["filename"].apply(lambda x: os.path.join(soundscape_dir, x))

# Remove missing files
ss_df = ss_df[ss_df["filepath"].apply(os.path.exists)].reset_index(drop=True)

print("Soundscape segments:", len(ss_df))


# =========================
# 3. LOAD TAXONOMY
# =========================
taxonomy = pd.read_csv("taxonomy.csv")
labels = taxonomy["primary_label"].values
label_to_idx = {label: i for i, label in enumerate(labels)}
idx_to_label = {i: label for label, i in label_to_idx.items()}
NUM_CLASSES = len(labels)
print("Total classes:", NUM_CLASSES)


# =========================
# 4. SPLIT BIRD CLIPS  (stratified by species)
# =========================
# Species with only 1 sample can't be stratified — pull them out first
counts = df["primary_label"].value_counts()
rare   = set(counts[counts < 2].index)

df_rare     = df[df["primary_label"].isin(rare)]
df_stratify = df[~df["primary_label"].isin(rare)]

bird_train_df, bird_val_df = train_test_split(
    df_stratify,
    test_size=0.2,
    stratify=df_stratify["primary_label"],
    random_state=SEED
)

# Rare species go entirely to train (can't validate what you barely have)
bird_train_df = pd.concat([bird_train_df, df_rare]).reset_index(drop=True)
bird_val_df   = bird_val_df.reset_index(drop=True)

print(f"\nBird clips  → train: {len(bird_train_df)}  val: {len(bird_val_df)}")


# =========================
# 5. SPLIT SOUNDSCAPES  (split by FILE, not by segment — avoids leakage)
# =========================
# Adjacent 5-second segments from the same file are highly correlated,
# so we group all segments from a file together before splitting.
unique_files = ss_df["filename"].unique()

ss_train_files, ss_val_files = train_test_split(
    unique_files,
    test_size=0.2,
    random_state=SEED
)

ss_train_df = ss_df[ss_df["filename"].isin(ss_train_files)].reset_index(drop=True)
ss_val_df   = ss_df[ss_df["filename"].isin(ss_val_files)].reset_index(drop=True)

print(f"Soundscapes → train files: {len(ss_train_files)}  val files: {len(ss_val_files)}")
print(f"             train segs:  {len(ss_train_df)}  val segs:  {len(ss_val_df)}")

print("\nUnique train_audio species:", df["primary_label"].nunique())
print("Species missing from train_audio:", NUM_CLASSES - df["primary_label"].nunique())

Files found: 12729
Soundscape segments: 1478
Total classes: 234

Bird clips  → train: 10183  val: 2546
Soundscapes → train files: 52  val files: 14
             train segs:  1160  val segs:  318

Unique train_audio species: 80
Species missing from train_audio: 154


In [3]:
def time_to_seconds(t):
    if isinstance(t, (int, float)):
        return float(t)
    if isinstance(t, str):
        h, m, s = t.split(":")
        return int(h) * 3600 + int(m) * 60 + float(s)
    raise ValueError(f"Invalid time format: {t}")

In [4]:
import numpy as np
import soundfile as sf
import librosa

TARGET_SR = 32000

def load_audio_segment(filepath, target_len, start_sec=None):
    try:
        audio, sr = sf.read(filepath)
    except:
        return np.zeros(target_len, dtype=np.float32)

    if audio.ndim == 2:
        audio = np.mean(audio, axis=1)

    if sr != TARGET_SR:
        audio = librosa.resample(audio, orig_sr=sr, target_sr=TARGET_SR)

    audio = audio.astype(np.float32)

    if start_sec is not None:
        start_sec = time_to_seconds(start_sec)
        start = int(start_sec * TARGET_SR)
        end   = start + target_len
        audio = audio[start:end]

    if len(audio) < target_len:
        audio = np.pad(audio, (0, target_len - len(audio)))
    else:
        audio = audio[:target_len]

    # ── Normalize so the backbone gets consistent input energy ──
    max_val = np.abs(audio).max()
    if max_val > 0:
        audio = audio / max_val          # peak normalize to [-1, 1]

    return audio


In [5]:
import torch
from torch.utils.data import Dataset

class BaseAudioDataset(Dataset):
    def __init__(self, df, label_to_idx, clip_duration, multi_label=False, augment=False):
        self.df           = df.reset_index(drop=True)
        self.label_to_idx = label_to_idx
        self.clip_len     = TARGET_SR * clip_duration
        self.multi_label  = multi_label
        self.augment      = augment

    def __len__(self):
        return len(self.df)

    def get_audio(self, row):
        return load_audio_segment(row["filepath"], self.clip_len)

    def get_label(self, row):
        label = torch.zeros(len(self.label_to_idx))
        if self.multi_label:
            for sp in str(row["primary_label"]).split(";"):
                if sp in self.label_to_idx:
                    label[self.label_to_idx[sp]] = 1.0
        else:
            if row["primary_label"] in self.label_to_idx:
                label[self.label_to_idx[row["primary_label"]]] = 1.0
        return label

    def _maybe_augment(self, audio):
        """Light waveform augmentation — only applied during training."""
        if not self.augment:
            return audio
        # Random gain
        gain  = np.random.uniform(0.6, 1.4)
        audio = audio * gain
        # Gaussian noise
        if np.random.rand() < 0.5:
            noise = np.random.randn(len(audio)).astype(np.float32)
            audio = audio + noise * np.random.uniform(0.001, 0.01)
        return np.clip(audio, -1.0, 1.0)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        audio = self.get_audio(row)
        audio = self._maybe_augment(audio)
        audio = torch.tensor(audio) 
        label = self.get_label(row)
        return audio, label

In [6]:
class BirdDataset(BaseAudioDataset):
    """Short XC/iNat clips — single primary label, 10-second clips."""
    def __init__(self, df, label_to_idx, augment=False):
        super().__init__(df, label_to_idx, clip_duration=10,
                         multi_label=False, augment=augment)

In [7]:
class SoundscapeDataset(BaseAudioDataset):
    """Labeled soundscape segments — multi-label, 5-second clips."""
    def __init__(self, df, label_to_idx, augment=False):
        super().__init__(df, label_to_idx, clip_duration=5,
                         multi_label=True, augment=augment)

    def get_audio(self, row):
        return load_audio_segment(
            row["filepath"],
            self.clip_len,
            start_sec=row["start"]
        )

In [8]:
from torch.utils.data import DataLoader
batch_size = 16
batch_size = 16
# ── Bird clip datasets (augment train only) ──────────────────
bird_train_ds = BirdDataset(bird_train_df, label_to_idx, augment=True)
bird_val_ds   = BirdDataset(bird_val_df,   label_to_idx, augment=False)

bird_train_loader = DataLoader(bird_train_ds, batch_size=batch_size, shuffle=True,
                               num_workers=1, pin_memory=batch_size, drop_last=True)
bird_val_loader   = DataLoader(bird_val_ds,   batch_size=16, shuffle=False,
                               num_workers=1, pin_memory=True)

# ── Soundscape datasets ──────────────────────────────────────
ss_train_ds = SoundscapeDataset(ss_train_df, label_to_idx, augment=True)
ss_val_ds   = SoundscapeDataset(ss_val_df,   label_to_idx, augment=False)

ss_train_loader = DataLoader(ss_train_ds, batch_size=16, shuffle=True,
                             num_workers=1, pin_memory=True, drop_last=True)
ss_val_loader   = DataLoader(ss_val_ds,   batch_size=16, shuffle=False,
                             num_workers=1, pin_memory=True)

print(f"Bird    — train batches: {len(bird_train_loader)}  val batches: {len(bird_val_loader)}")
print(f"Soundsc — train batches: {len(ss_train_loader)}  val batches: {len(ss_val_loader)}")

Bird    — train batches: 636  val batches: 160
Soundsc — train batches: 72  val batches: 20


In [9]:
import sys
sys.path.append("/home/users/ss1482/sangcs372final/audioset_tagging_cnn")
sys.path.append("/home/users/ss1482/sangcs372final/audioset_tagging_cnn/pytorch")

from pytorch.models import Cnn14

In [10]:
model = Cnn14(
    sample_rate=32000,
    window_size=1024,
    hop_size=320,
    mel_bins=64,
    fmin=50,
    fmax=14000,
    classes_num=527   # original AudioSet head — will be replaced below
)

In [11]:
checkpoint = torch.load("Cnn14_mAP=0.431.pth", map_location="cpu")
model.load_state_dict(checkpoint["model"], strict=False)
print("Pretrained weights loaded.")

Pretrained weights loaded.


In [12]:
# Replace AudioSet head (527 classes) with our Pantanal head (234 classes)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.fc_audioset = torch.nn.Sequential(
    torch.nn.Linear(2048, 512),
    torch.nn.BatchNorm1d(512),   # normalizes the sparse fc1 output
    torch.nn.GELU(),             # GELU doesn't hard-zero negatives like ReLU does
    torch.nn.Dropout(0.3),
    torch.nn.Linear(512, NUM_CLASSES)
).to(device)


model  = model.to(device)
print("Model ready on:", device)

Model ready on: cuda


In [13]:
def freeze_backbone_partial(model):
    for name, param in model.named_parameters():
        param.requires_grad = False

    for name, param in model.named_parameters():
        if any(x in name for x in ["conv_block4", "conv_block5", "conv_block6", "fc1", "fc_audioset"]):
            param.requires_grad = True

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Partial freeze. Trainable params: {trainable:,}")
def unfreeze_all(model):
    """Unfreeze every layer for full fine-tuning."""
    for param in model.parameters():
        param.requires_grad = True
    total = sum(p.numel() for p in model.parameters())
    print(f"All layers unfrozen. Total trainable params: {total:,}")

In [22]:
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score

def run_validation(model, loader, criterion, device):
    """Returns average val loss, macro ROC-AUC, and per-class AUC dictionary."""
    model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for audio, label in loader:
            audio, label = audio.to(device), label.to(device)
            output = model(audio)
            logits = output["clipwise_output"] if isinstance(output, dict) else output
            loss = criterion(logits, label)
            total_loss += loss.item()
            
            all_preds.append(torch.sigmoid(logits).cpu().numpy())
            all_labels.append(label.cpu().numpy())

    all_preds = np.concatenate(all_preds, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)

    # 1. Calculate Per-Class AUC
    per_class_auc = {}
    n_classes = all_labels.shape[1]
    
    for i in range(n_classes):
        # We can only calculate AUC if the class has both positive and negative samples in the val set
        if len(np.unique(all_labels[:, i])) > 1:
            score = roc_auc_score(all_labels[:, i], all_preds[:, i])
            per_class_auc[i] = score
        else:
            per_class_auc[i] = np.nan # Not enough data in this split for this bird

    # 2. Calculate Macro AUC (ignoring NaNs)
    valid_scores = [v for v in per_class_auc.values() if not np.isnan(v)]
    macro_auc = np.mean(valid_scores) if valid_scores else 0.0

    return total_loss / len(loader), macro_auc, per_class_auc

## Phase 1 — Pre-train on Bird Clips (backbone frozen)

Only the new 234-class head is trained here.  
The CNN14 backbone keeps its AudioSet weights and learns nothing yet — this prevents catastrophic forgetting while the head finds reasonable initialisation.

In [15]:
import torch.nn as nn
import torch.nn.functional as F

import torch
import torch.nn as nn
import torch.nn.functional as F

class Cnn14Pantanal(nn.Module):
    def __init__(self, base_model, num_classes):
        super().__init__()
        # Copy all layers from the pretrained model
        self.spectrogram_extractor = base_model.spectrogram_extractor
        self.logmel_extractor      = base_model.logmel_extractor
        self.spec_augmenter        = base_model.spec_augmenter
        self.bn0         = base_model.bn0
        self.conv_block1 = base_model.conv_block1
        self.conv_block2 = base_model.conv_block2
        self.conv_block3 = base_model.conv_block3
        self.conv_block4 = base_model.conv_block4
        self.conv_block5 = base_model.conv_block5
        self.conv_block6 = base_model.conv_block6
        
        # FIX: Wrap fc1 in Sequential to match the saved state_dict "fc1.0.weight"
        self.fc1 = nn.Sequential(
            base_model.fc1  # This keeps the pretrained weights as the 0th element
        )
        
        self.gelu = nn.GELU() 

        # New head for 234 classes
        self.fc_audioset = nn.Sequential(
            nn.Linear(2048, 512),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, input):
        x = self.spectrogram_extractor(input)
        x = self.logmel_extractor(x)

        x = x.transpose(1, 3)
        x = self.bn0(x)
        x = x.transpose(1, 3)

        if self.training:
            x = self.spec_augmenter(x)

        x = self.conv_block1(x, pool_size=(2, 2), pool_type='avg')
        x = F.dropout(x, p=0.2, training=self.training)
        x = self.conv_block2(x, pool_size=(2, 2), pool_type='avg')
        x = F.dropout(x, p=0.2, training=self.training)
        x = self.conv_block3(x, pool_size=(2, 2), pool_type='avg')
        x = F.dropout(x, p=0.2, training=self.training)
        x = self.conv_block4(x, pool_size=(2, 2), pool_type='avg')
        x = F.dropout(x, p=0.2, training=self.training)
        x = self.conv_block5(x, pool_size=(2, 2), pool_type='avg')
        x = F.dropout(x, p=0.2, training=self.training)
        x = self.conv_block6(x, pool_size=(1, 1), pool_type='avg')
        x = F.dropout(x, p=0.2, training=self.training)

        x = torch.mean(x, dim=3)
        (x1, _) = torch.max(x, dim=2)
        x2 = torch.mean(x, dim=2)
        x = x1 + x2

        x = F.dropout(x, p=0.5, training=self.training)
        
        # Because fc1 is now Sequential, we call it directly. 
        # The weight is now internally stored at self.fc1[0].weight
        x = self.gelu(self.fc1(x)) 
        
        x = F.dropout(x, p=0.5, training=self.training)
        logits = self.fc_audioset(x)

        return {"clipwise_output": logits, "embedding": x}
# ── Wrap the existing model ──────────────────────────────────
model = Cnn14Pantanal(model, NUM_CLASSES).to(device)

# ── Verify fc1 zeros are gone ────────────────────────────────
activations = {}
def hook_fn(module, input, output):
    activations["fc1"] = output.detach()

hook = model.fc1.register_forward_hook(hook_fn)
audio, label = next(iter(bird_train_loader))
with torch.no_grad():
    model(audio.to(device))
hook.remove()

print("fc1 output mean:", activations["fc1"].mean().item())
print("fc1 output std: ", activations["fc1"].std().item())
print("fc1 zeros:      ", (activations["fc1"] == 0).float().mean().item())

fc1 output mean: -0.31239742040634155
fc1 output std:  0.634048581123352
fc1 zeros:       0.0


In [16]:
import os
import torch.optim as optim

# ── Hyperparameters ──────────────────────────────────────────
PHASE1_EPOCHS  = 10
WEIGHT_DECAY   = 1e-2
BATCH_SIZE     = 8
WARMUP_EPOCHS  = 0
PATIENCE       = 3
MIN_DELTA      = 1e-4
RESUME         = True   # ← NEW

# ── Early stopping state ─────────────────────────────────────
class EarlyStopping:
    def __init__(self, patience=PATIENCE, min_delta=MIN_DELTA):
        self.patience   = patience
        self.min_delta  = min_delta
        self.counter    = 0
        self.best_auc   = 0.0
        self.should_stop = False

    def step(self, val_auc):
        if val_auc > self.best_auc + self.min_delta:
            self.best_auc  = val_auc
            self.counter   = 0
        else:
            self.counter  += 1
            print(f"  ↳ No improvement for {self.counter}/{self.patience} epochs")
            if self.counter >= self.patience:
                self.should_stop = True
                print("  ✗ Early stopping triggered.")

# Freeze backbone
freeze_backbone_partial(model)

criterion = torch.nn.BCEWithLogitsLoss()

optimizer = optim.AdamW([
    {"params": [p for n,p in model.named_parameters() if "conv_block4" in n], "lr": 1e-5},
    {"params": [p for n,p in model.named_parameters() if "conv_block5" in n], "lr": 5e-5},
    {"params": [p for n,p in model.named_parameters() if "conv_block6" in n], "lr": 1e-4},
    {"params": [p for n,p in model.named_parameters() if "fc1" in n],         "lr": 2e-4},
    {"params": [p for n,p in model.named_parameters() if "fc_audioset" in n], "lr": 5e-4},
], weight_decay=WEIGHT_DECAY)

def get_lr_scale(epoch):
    if epoch < WARMUP_EPOCHS:
        return (epoch + 1) / max(1, WARMUP_EPOCHS)
    progress = (epoch - WARMUP_EPOCHS) / max(1, PHASE1_EPOCHS - WARMUP_EPOCHS)
    return 0.1 + 0.9 * 0.5 * (1 + torch.cos(torch.tensor(3.14159 * progress)).item())

scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=get_lr_scale)

early_stopper = EarlyStopping(patience=PATIENCE, min_delta=MIN_DELTA)

Partial freeze. Trainable params: 79,698,666


In [17]:
# Run ONE forward+backward pass and check if gradients are actually flowing
model.train()
audio, label = next(iter(bird_train_loader))
audio = audio.to(device)
label = label.to(device)

optimizer.zero_grad()
output = model(audio)
logits = output["clipwise_output"] if isinstance(output, dict) else output
loss   = criterion(logits, label)
loss.backward()

# Check gradient magnitudes on the head and a mid-level layer
for name, param in model.named_parameters():
    if param.grad is not None and param.grad.abs().max() > 0:
        print(f"{name:50s}  grad_max={param.grad.abs().max():.6f}")

conv_block4.conv1.weight                            grad_max=0.000324
conv_block4.conv2.weight                            grad_max=0.000153
conv_block4.bn1.weight                              grad_max=0.001375
conv_block4.bn1.bias                                grad_max=0.000311
conv_block4.bn2.weight                              grad_max=0.001201
conv_block4.bn2.bias                                grad_max=0.000413
conv_block5.conv1.weight                            grad_max=0.000055
conv_block5.conv2.weight                            grad_max=0.000045
conv_block5.bn1.weight                              grad_max=0.000656
conv_block5.bn1.bias                                grad_max=0.000294
conv_block5.bn2.weight                              grad_max=0.000472
conv_block5.bn2.bias                                grad_max=0.000291
conv_block6.conv1.weight                            grad_max=0.000047
conv_block6.conv2.weight                            grad_max=0.000050
conv_block6.bn1.weig

In [25]:


# ── Resume logic ─────────────────────────────────────────────
start_epoch = 1
best_phase1_auc = 0.0

if RESUME and os.path.exists("bestbest_phase1.pth"):
    ckpt = torch.load("bestbest_phase1.pth", map_location=device)
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])

    if "scheduler" in ckpt:
        scheduler.load_state_dict(ckpt["scheduler"])

    start_epoch = ckpt["epoch"] + 1
    best_phase1_auc = ckpt["val_auc"]

    print(f"Resuming from epoch {start_epoch-1}, best AUC {best_phase1_auc:.4f}")

# ── Training loop ────────────────────────────────────────────
for epoch in range(start_epoch, PHASE1_EPOCHS + 1):

    model.train()
    running_loss = 0.0

    for batch_idx, (audio, label) in enumerate(bird_train_loader):
        audio = audio.to(device)
        label = label.to(device)

        optimizer.zero_grad()
        output = model(audio)
        logits = output["clipwise_output"] if isinstance(output, dict) else output
        loss   = criterion(logits, label)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        running_loss += loss.item()
        if (batch_idx + 1) % 50 == 0:
            current_lr = optimizer.param_groups[0]["lr"]
            print(f"  [P1 E{epoch} step {batch_idx+1}] "
                  f"loss: {running_loss/(batch_idx+1):.4f}  lr: {current_lr:.2e}")

    scheduler.step()

    # ── validate ─────────────────────────────────────────────
    val_loss, val_auc = run_validation(model, bird_val_loader, criterion, device)
    current_lr = optimizer.param_groups[0]["lr"]

    print(f"Epoch {epoch}/{PHASE1_EPOCHS}  "
          f"train_loss={running_loss/len(bird_train_loader):.4f}  "
          f"val_loss={val_loss:.4f}  val_auc={val_auc:.4f}  "
          f"lr={current_lr:.2e}")

    # ── save latest (NEW, optional but useful) ────────────────
    torch.save({
        "epoch": epoch,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "val_auc": val_auc,
        "batch_size": BATCH_SIZE,
    }, "latest_phase1.pth")

    # ── checkpoint if best ───────────────────────────────────
    if val_auc > best_phase1_auc:
        best_phase1_auc = val_auc
        torch.save({
            "epoch": epoch,
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),  # ← added
            "val_auc": val_auc,
            "batch_size": BATCH_SIZE,
        }, "bestbest_phase1.pth")
        print("  ✓ Saved best Phase 1 checkpoint")

    # ── early stopping ───────────────────────────────────────
    early_stopper.step(val_auc)
    if early_stopper.should_stop:
        print(f"\nStopped early at epoch {epoch}. "
              f"Best val AUC: {best_phase1_auc:.4f}")
        break

else:
    print(f"\nPhase 1 complete. Best val AUC: {best_phase1_auc:.4f}")

# ── safe reload ──────────────────────────────────────────────
if os.path.exists("bestbest_phase1.pth"):
    print("Reloading best Phase 1 weights...")
    model.load_state_dict(torch.load("best_phase1.pth")["model"])
else:
    print("No checkpoint found to reload.")


Resuming from epoch 10, best AUC 0.9817

Phase 1 complete. Best val AUC: 0.9817
Reloading best Phase 1 weights...


In [ ]:
ckpt = torch.load("best_phase1.pth", map_location=device)
state_dict = ckpt["model"]

# --- fix fc1 mismatch ---
fixed_state_dict = {}
for k, v in state_dict.items():
    if k.startswith("fc1.0."):
        new_key = k.replace("fc1.0.", "fc1.")
    else:
        new_key = k
    fixed_state_dict[new_key] = v

# load weights
model.load_state_dict(fixed_state_dict)

model.to(device)
model.eval()

## Phase 2 — Fine-tune on Labeled Soundscapes (all layers unfrozen)

Now we unfreeze the backbone and train on the real field recordings.  
We use **discriminative learning rates**: the backbone gets a much lower LR than the head to avoid destroying what it learned in Phase 1.

In [18]:
def mixup_data(x, y, alpha=0.2):
    '''Returns mixed inputs, pairs of targets, and lambda'''
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1

    batch_size = x.size()[0]
    index = torch.randperm(batch_size).to(x.device)

    mixed_x = lam * x + (1 - lam) * x[index, :]
    mixed_y = lam * y + (1 - lam) * y[index, :]
    return mixed_x, mixed_y

In [ ]:
import torch
import torch.optim as optim
import torch.nn.functional as F
import matplotlib.pyplot as plt
import os
import pandas as pd
import numpy as np

# --- 1. Focal Loss Definition ---
class FocalLoss(torch.nn.Module):
    def __init__(self, alpha=1, gamma=2):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        # inputs are logits, targets are multi-label 0/1
        bce_loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        pt = torch.exp(-bce_loss) # probability of the correct class
        f_loss = self.alpha * (1 - pt)**self.gamma * bce_loss
        return f_loss.mean()

# --- 2. Setup Phase 2 ---
PHASE2_EPOCHS = 30  
BACKBONE_LR   = 1e-5  
HEAD_LR       = 1e-4
PATIENCE      = 5
MIXUP_ALPHA   = 0.2
ACCUM_STEPS   = 4  # Gradient accumulation: update every 4 batches

early_stopper = EarlyStopping(patience=PATIENCE, mode='max')
criterion = FocalLoss(gamma=2.0) # Focal Loss to help with rare/hard classes

train_losses, val_losses, val_aucs = [], [], []

# Separate parameters for discriminative learning rates
head_params     = list(model.fc_audioset.parameters())
head_ids        = set(id(p) for p in head_params)
backbone_params = [p for p in model.parameters() if id(p) not in head_ids]

optimizer = optim.AdamW([
    {"params": backbone_params, "lr": BACKBONE_LR, "weight_decay": 1e-4},
    {"params": head_params,      "lr": HEAD_LR,       "weight_decay": 1e-3},
])

scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)

# --- 3. Training Loop ---
print("Starting Phase 2 (Focal Loss + Accumulation)...")
for epoch in range(1, PHASE2_EPOCHS + 1):
    model.train()
    running_loss = 0.0
    optimizer.zero_grad() # Zero gradients once at the start of epoch

    for i, (audio, label) in enumerate(ss_train_loader):
        audio, label = audio.to(device), label.to(device)
        
        # Apply mixup
        audio, label = mixup_data(audio, label, alpha=MIXUP_ALPHA)
        
        output = model(audio)
        logits = output["clipwise_output"] if isinstance(output, dict) else output
        
        # Calculate loss and scale by accumulation steps
        loss = criterion(logits, label)
        (loss / ACCUM_STEPS).backward()
        
        # Update weights only every ACCUM_STEPS
        if (i + 1) % ACCUM_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            optimizer.zero_grad()
            
        running_loss += loss.item()

    # Validation step
    avg_train_loss = running_loss / len(ss_train_loader)
    val_loss, val_auc, per_class_auc = run_validation(model, ss_val_loader, criterion, device)
    
    train_losses.append(avg_train_loss)
    val_losses.append(val_loss)
    val_aucs.append(val_auc)

    # Step scheduler and early stopper
    scheduler.step(val_auc)
    early_stopper(val_auc)
    
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch {epoch} | Loss: {avg_train_loss:.4f} | Val AUC: {val_auc:.4f} | LR: {current_lr:.2e}")

    # Monitor Per-Class performance
    valid_scores = [(idx, score) for idx, score in per_class_auc.items() if not np.isnan(score)]
    sorted_auc = sorted(valid_scores, key=lambda x: x[1], reverse=True)
    
    if sorted_auc:
        print(f"  > Top 3 Birds: {[(idx_to_label[idx], f'{s:.3f}') for idx, s in sorted_auc[:3]]}")
        print(f"  > Bottom 3 Birds: {[(idx_to_label[idx], f'{s:.3f}') for idx, s in sorted_auc[-3:]]}")

    # Save Best Model and CSV
    if val_auc == max(val_aucs):
        torch.save({"model": model.state_dict(), "auc": val_auc}, "best_phase2.pth")
        
        # Save readable bird results to CSV
        results_data = [{"Bird_Name": idx_to_label[idx], "AUC": score} for idx, score in per_class_auc.items()]
        pd.DataFrame(results_data).to_csv('best_per_class_auc.csv', index=False)
        print("  ✓ Saved best model and updated AUC CSV")

    if early_stopper.early_stop:
        print(f"Early stopping at epoch {epoch}")
        break

# --- 4. Plot curves ---
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Train')
plt.plot(val_losses, label='Val')
plt.title('Loss Curves (Focal Loss)')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(val_aucs, color='green', label='Val AUC')
plt.title('AUC History')
plt.show()

Starting Phase 2 (Focal Loss + Accumulation)...
Epoch 1 | Loss: 0.0069 | Val AUC: 0.8299 | LR: 1.00e-05
  > Top 3 Birds: [('23158', '1.000'), ('whtdov', '0.997'), ('47158son17', '0.992')]
  > Bottom 3 Birds: [('47158son03', '0.535'), ('undtin1', '0.505'), ('ruther1', '0.443')]
  ✓ Saved best model and updated AUC CSV
EarlyStopping counter: 1 out of 5
Epoch 2 | Loss: 0.0072 | Val AUC: 0.8290 | LR: 1.00e-05
  > Top 3 Birds: [('23158', '1.000'), ('whtdov', '0.996'), ('47158son07', '0.992')]
  > Bottom 3 Birds: [('undtin1', '0.528'), ('47158son03', '0.515'), ('ruther1', '0.424')]
EarlyStopping counter: 2 out of 5
Epoch 3 | Loss: 0.0072 | Val AUC: 0.8254 | LR: 1.00e-05
  > Top 3 Birds: [('23158', '1.000'), ('whtdov', '0.997'), ('47158son07', '0.995')]
  > Bottom 3 Birds: [('undtin1', '0.536'), ('47158son03', '0.489'), ('ruther1', '0.437')]
Improvement detected! Counter reset.
Epoch 4 | Loss: 0.0071 | Val AUC: 0.8345 | LR: 1.00e-05
  > Top 3 Birds: [('23158', '1.000'), ('whtdov', '0.996'), (

In [ ]:
# Quick sanity check — one batch through the final model
model.load_state_dict(torch.load("best_phase2.pth", map_location=device))
model.eval()

audio, label = next(iter(ss_val_loader))
audio = audio.to(device)

with torch.no_grad():
    output = model(audio)
    logits = output["clipwise_output"] if isinstance(output, dict) else output
    probs  = torch.sigmoid(logits)

print("Output shape:", probs.shape)   # should be (batch, 234)
print("Min prob:", probs.min().item(), "Max prob:", probs.max().item())